This is a companion notebook for the book [Deep Learning with Python, Third Edition](https://www.manning.com/books/deep-learning-with-python-third-edition). For readability, it only contains runnable code blocks and section titles, and omits everything else in the book: text paragraphs, figures, and pseudocode.

**If you want to be able to follow what's going on, I recommend reading the notebook side by side with your copy of the book.**

The book's contents are available online at [deeplearningwithpython.io](https://deeplearningwithpython.io).

In [ ]:
!pip install torch torchvision --upgrade -q

## Text classification

### A brief history of natural language processing

### Preparing text data

In [0]:
import regex as re

def split_chars(text):
    return re.findall(r".", text)

In [0]:
chars = split_chars("The quick brown fox jumped over the lazy dog.")
chars[:12]

In [0]:
def split_words(text):
    return re.findall(r"[\w]+|[.,!?;]", text)

In [0]:
split_words("The quick brown fox jumped over the dog.")

In [0]:
vocabulary = {
    "[UNK]": 0,
    "the": 1,
    "quick": 2,
    "brown": 3,
    "fox": 4,
    "jumped": 5,
    "over": 6,
    "dog": 7,
    ".": 8,
}
words = split_words("The quick brown fox jumped over the lazy dog.")
indices = [vocabulary.get(word, 0) for word in words]

#### Character and word tokenization

In [0]:
class CharTokenizer:
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        return re.findall(r".", inputs)

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

In [0]:
import collections

def compute_char_vocabulary(inputs, max_size):
    char_counts = collections.Counter()
    for x in inputs:
        x = x.lower()
        tokens = re.findall(r".", x)
        char_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = char_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

In [0]:
class WordTokenizer:
    def __init__(self, vocabulary):
        self.vocabulary = vocabulary
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def split(self, inputs):
        return re.findall(r"[\w]+|[.,!?;]", inputs)

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

In [0]:
def compute_word_vocabulary(inputs, max_size):
    word_counts = collections.Counter()
    for x in inputs:
        x = x.lower()
        tokens = re.findall(r"[\w]+|[.,!?;]", x)
        word_counts.update(tokens)
    vocabulary = ["[UNK]"]
    most_common = word_counts.most_common(max_size - len(vocabulary))
    for token, count in most_common:
        vocabulary.append(token)
    return dict((token, i) for i, token in enumerate(vocabulary))

In [ ]:
# PyTorch: no keras.utils.get_file; download Moby Dick with urllib instead.
import os, urllib.request

filename = "moby10b.txt"
if not os.path.exists(filename):
    urllib.request.urlretrieve(
        "https://www.gutenberg.org/files/2701/old/moby10b.txt", filename
    )
moby_dick = list(open(filename, "r"))

vocabulary = compute_char_vocabulary(moby_dick, max_size=100)
char_tokenizer = CharTokenizer(vocabulary)

In [0]:
print("Vocabulary length:", len(vocabulary))

In [0]:
print("Vocabulary start:", list(vocabulary.keys())[:10])

In [0]:
print("Vocabulary end:", list(vocabulary.keys())[-10:])

In [0]:
print("Line length:", len(char_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

In [0]:
vocabulary = compute_word_vocabulary(moby_dick, max_size=2_000)
word_tokenizer = WordTokenizer(vocabulary)

In [0]:
print("Vocabulary length:", len(vocabulary))

In [0]:
print("Vocabulary start:", list(vocabulary.keys())[:5])

In [0]:
print("Vocabulary end:", list(vocabulary.keys())[-5:])

In [0]:
print("Line length:", len(word_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

#### Subword tokenization

In [0]:
data = [
    "the quick brown fox",
    "the slow brown fox",
    "the quick brown foxhound",
]

In [0]:
def count_and_split_words(data):
    counts = collections.Counter()
    for line in data:
        line = line.lower()
        for word in re.findall(r"[\w]+|[.,!?;]", line):
            chars = re.findall(r".", word)
            split_word = " ".join(chars)
            counts[split_word] += 1
    return dict(counts)

counts = count_and_split_words(data)

In [0]:
counts

In [0]:
def count_pairs(counts):
    pairs = collections.Counter()
    for word, freq in counts.items():
        symbols = word.split()
        for pair in zip(symbols[:-1], symbols[1:]):
            pairs[pair] += freq
    return pairs

def merge_pair(counts, first, second):
    split = re.compile(f"(?<!\S){first} {second}(?!\S)")
    merged = f"{first}{second}"
    return {split.sub(merged, word): count for word, count in counts.items()}

for i in range(10):
    pairs = count_pairs(counts)
    first, second = max(pairs, key=pairs.get)
    counts = merge_pair(counts, first, second)
    print(list(counts.keys()))

In [0]:
def compute_sub_word_vocabulary(dataset, vocab_size):
    counts = count_and_split_words(dataset)

    char_counts = collections.Counter()
    for word in counts:
        for char in word.split():
            char_counts[char] += counts[word]
    most_common = char_counts.most_common()
    vocab = ["[UNK]"] + [char for char, freq in most_common]
    merges = []

    while len(vocab) < vocab_size:
        pairs = count_pairs(counts)
        if not pairs:
            break
        first, second = max(pairs, key=pairs.get)
        counts = merge_pair(counts, first, second)
        vocab.append(f"{first}{second}")
        merges.append(f"{first} {second}")

    vocab = dict((token, index) for index, token in enumerate(vocab))
    merges = dict((token, rank) for rank, token in enumerate(merges))
    return vocab, merges

In [0]:
class SubWordTokenizer:
    def __init__(self, vocabulary, merges):
        self.vocabulary = vocabulary
        self.merges = merges
        self.unk_id = vocabulary["[UNK]"]

    def standardize(self, inputs):
        return inputs.lower()

    def bpe_merge(self, word):
        while True:
            pairs = re.findall(r"(?<!\S)\S+ \S+(?!\S)", word, overlapped=True)
            if not pairs:
                break
            best = min(pairs, key=lambda pair: self.merges.get(pair, 1e9))
            if best not in self.merges:
                break
            first, second = best.split()
            split = re.compile(f"(?<!\S){first} {second}(?!\S)")
            merged = f"{first}{second}"
            word = split.sub(merged, word)
        return word

    def split(self, inputs):
        tokens = []
        for word in re.findall(r"[\w]+|[.,!?;]", inputs):
            word = " ".join(re.findall(r".", word))
            word = self.bpe_merge(word)
            tokens.extend(word.split())
        return tokens

    def index(self, tokens):
        return [self.vocabulary.get(t, self.unk_id) for t in tokens]

    def __call__(self, inputs):
        inputs = self.standardize(inputs)
        tokens = self.split(inputs)
        indices = self.index(tokens)
        return indices

In [0]:
vocabulary, merges = compute_sub_word_vocabulary(moby_dick, 2_000)
sub_word_tokenizer = SubWordTokenizer(vocabulary, merges)

In [0]:
print("Vocabulary length:", len(vocabulary))

In [0]:
print("Vocabulary start:", list(vocabulary.keys())[:10])

In [0]:
print("Vocabulary end:", list(vocabulary.keys())[-7:])

In [0]:
print("Line length:", len(sub_word_tokenizer(
   "Call me Ishmael. Some years ago--never mind how long precisely."
)))

### Sets vs. sequences

#### Loading the IMDb classification dataset

In [ ]:
# PyTorch: download and extract the aclImdb dataset manually (no keras get_file).
import os, pathlib, shutil, random, tarfile, urllib.request

archive = "aclImdb_v1.tar.gz"
if not os.path.exists(archive):
    urllib.request.urlretrieve(
        "https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz", archive
    )
if not os.path.exists("aclImdb"):
    with tarfile.open(archive) as tar:
        tar.extractall()

imdb_extract_dir = pathlib.Path("aclImdb")

In [0]:
for path in imdb_extract_dir.glob("*/*"):
    if path.is_dir():
        print(path)

In [0]:
print(open(imdb_extract_dir / "train" / "pos" / "4077_10.txt", "r").read())

In [0]:
train_dir = pathlib.Path("imdb_train")
test_dir = pathlib.Path("imdb_test")
val_dir = pathlib.Path("imdb_val")

shutil.copytree(imdb_extract_dir / "test", test_dir)

val_percentage = 0.2
for category in ("neg", "pos"):
    src_dir = imdb_extract_dir / "train" / category
    src_files = os.listdir(src_dir)
    random.Random(1337).shuffle(src_files)
    num_val_samples = int(len(src_files) * val_percentage)

    os.makedirs(val_dir / category)
    for file in src_files[:num_val_samples]:
        shutil.copy(src_dir / file, val_dir / category / file)
    os.makedirs(train_dir / category)
    for file in src_files[num_val_samples:]:
        shutil.copy(src_dir / file, train_dir / category / file)

In [ ]:
# PyTorch: there's no text_dataset_from_directory. We build a Dataset that reads
# the raw .txt files from the neg/pos subdirectories (label 0/1, sorted like Keras),
# and wrap each split in a DataLoader. Batches are (list[str] texts, float label tensor).
import torch
from torch.utils.data import Dataset, DataLoader

class TextFolderDataset(Dataset):
    def __init__(self, directory):
        self.samples = []
        directory = pathlib.Path(directory)
        for label, category in enumerate(("neg", "pos")):
            for path in sorted((directory / category).glob("*.txt")):
                self.samples.append((path, float(label)))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        text = open(path, "r").read()
        return text, label

def text_collate(batch):
    texts = [t for t, _ in batch]
    labels = torch.tensor([y for _, y in batch], dtype=torch.float32)
    return texts, labels

batch_size = 32
train_ds = DataLoader(
    TextFolderDataset(train_dir), batch_size=batch_size, shuffle=True,
    collate_fn=text_collate,
)
val_ds = DataLoader(
    TextFolderDataset(val_dir), batch_size=batch_size, collate_fn=text_collate,
)
test_ds = DataLoader(
    TextFolderDataset(test_dir), batch_size=batch_size, collate_fn=text_collate,
)

### Set models

#### Training a bag-of-words model

In [ ]:
# PyTorch: there's no layers.TextVectorization. We reimplement it as a small class.
# adapt() builds a frequency-ranked vocabulary (slot 0 = "", slot 1 = "[UNK]" like
# Keras), and __call__ produces multi-hot vectors. ngrams=2 enables bigram features.
import collections

class TextVectorization:
    def __init__(self, max_tokens, output_mode="multi_hot",
                 output_sequence_length=None, ngrams=1, vocabulary=None):
        self.max_tokens = max_tokens
        self.output_mode = output_mode
        self.output_sequence_length = output_sequence_length
        self.ngrams = ngrams
        self.vocab = None
        self.index = None
        if vocabulary is not None:
            self.set_vocabulary(vocabulary)

    def _tokens(self, text):
        words = text.lower().split()
        if self.ngrams and self.ngrams >= 2:
            grams = list(words)
            for n in range(2, self.ngrams + 1):
                grams += [" ".join(words[i:i + n]) for i in range(len(words) - n + 1)]
            return grams
        return words

    def adapt(self, texts):
        counts = collections.Counter()
        for text in texts:
            counts.update(self._tokens(text))
        # Reserve slot 0 for padding ("") and slot 1 for "[UNK]", matching Keras.
        most_common = counts.most_common(self.max_tokens - 2)
        self.vocab = ["", "[UNK]"] + [tok for tok, _ in most_common]
        self.set_vocabulary(self.vocab)

    def set_vocabulary(self, vocabulary):
        self.vocab = list(vocabulary)
        self.index = {tok: i for i, tok in enumerate(self.vocab)}

    def get_vocabulary(self):
        return self.vocab

    def __call__(self, texts):
        if isinstance(texts, str):
            texts = [texts]
        if self.output_mode == "multi_hot":
            out = torch.zeros((len(texts), self.max_tokens), dtype=torch.float32)
            for r, text in enumerate(texts):
                for tok in self._tokens(text):
                    out[r, self.index.get(tok, 1)] = 1.0
            return out
        else:  # "int"
            seq_len = self.output_sequence_length
            out = torch.zeros((len(texts), seq_len), dtype=torch.long)
            for r, text in enumerate(texts):
                ids = [self.index.get(tok, 1) for tok in self._tokens(text)][:seq_len]
                out[r, :len(ids)] = torch.tensor(ids, dtype=torch.long)
            return out

max_tokens = 20_000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="multi_hot",
)
# adapt over all training texts.
train_texts = [open(p, "r").read() for p, _ in TextFolderDataset(train_dir).samples]
text_vectorization.adapt(train_texts)

# PyTorch: instead of dataset.map, we vectorize inside the training loop's collate.
# These helper DataLoaders yield (multi_hot_tensor, label_tensor).
def make_vectorized_loader(folder, shuffle=False):
    base = TextFolderDataset(folder)
    def collate(batch):
        texts = [t for t, _ in batch]
        labels = torch.tensor([y for _, y in batch], dtype=torch.float32)
        return text_vectorization(texts), labels
    return DataLoader(base, batch_size=batch_size, shuffle=shuffle, collate_fn=collate)

bag_of_words_train_ds = make_vectorized_loader(train_dir, shuffle=True)
bag_of_words_val_ds = make_vectorized_loader(val_dir)
bag_of_words_test_ds = make_vectorized_loader(test_dir)

In [ ]:
x, y = next(iter(bag_of_words_train_ds))
x.shape

In [ ]:
y.shape

In [ ]:
# PyTorch: a linear classifier is a single nn.Linear. We drop the sigmoid and use
# BCEWithLogitsLoss (it applies the sigmoid internally). compile() becomes a helper
# that returns the model plus its optimizer and loss function.
from torch import nn

def build_linear_classifier(max_tokens, name):
    model = nn.Linear(max_tokens, 1)
    model.name = name
    optimizer = torch.optim.Adam(model.parameters())
    loss_fn = nn.BCEWithLogitsLoss()
    return model, optimizer, loss_fn

model, optimizer, loss_fn = build_linear_classifier(max_tokens, "bag_of_words_classifier")

In [ ]:
# PyTorch: no model.summary(); printing the module lists its layers and shapes.
print(model)

In [ ]:
# PyTorch: model.fit + EarlyStopping become an explicit loop. We track validation
# loss, keep the best weights, and stop after `patience` epochs without improvement.
import copy

def evaluate(model, loss_fn, loader):
    model.eval()
    total_loss, correct, n = 0.0, 0, 0
    with torch.no_grad():
        for inputs, targets in loader:
            logits = model(inputs).squeeze(-1)
            total_loss += loss_fn(logits, targets).item() * len(targets)
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == targets).sum().item()
            n += len(targets)
    return total_loss / n, correct / n

def fit(model, optimizer, loss_fn, train_loader, val_loader, epochs=10, patience=2):
    history = {"accuracy": [], "val_accuracy": []}
    best_val_loss, best_state, wait = float("inf"), None, 0
    for epoch in range(epochs):
        model.train()
        correct, n = 0, 0
        for inputs, targets in train_loader:
            logits = model(inputs).squeeze(-1)
            loss = loss_fn(logits, targets)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            preds = (torch.sigmoid(logits) > 0.5).float()
            correct += (preds == targets).sum().item()
            n += len(targets)
        train_acc = correct / n
        val_loss, val_acc = evaluate(model, loss_fn, val_loader)
        history["accuracy"].append(train_acc)
        history["val_accuracy"].append(val_acc)
        print(f"Epoch {epoch}: acc {train_acc:.4f} val_loss {val_loss:.4f} val_acc {val_acc:.4f}")
        # EarlyStopping(restore_best_weights=True, patience=2) on val_loss.
        if val_loss < best_val_loss:
            best_val_loss, best_state, wait = val_loss, copy.deepcopy(model.state_dict()), 0
        else:
            wait += 1
            if wait >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return history

history = fit(
    model, optimizer, loss_fn,
    bag_of_words_train_ds, bag_of_words_val_ds,
    epochs=10, patience=2,
)

In [ ]:
import matplotlib.pyplot as plt

# PyTorch: our fit() returns a plain dict, so we read history[...] directly.
accuracy = history["accuracy"]
val_accuracy = history["val_accuracy"]
epochs = range(1, len(accuracy) + 1)

plt.plot(epochs, accuracy, "r--", label="Training accuracy")
plt.plot(epochs, val_accuracy, "b", label="Validation accuracy")
plt.title("Training and validation accuracy")
plt.legend()
plt.show()

In [ ]:
test_loss, test_acc = evaluate(model, loss_fn, bag_of_words_test_ds)
test_acc

#### Training a bigram model

In [ ]:
max_tokens = 30_000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="multi_hot",
    ngrams=2,
)
text_vectorization.adapt(train_texts)

bigram_train_ds = make_vectorized_loader(train_dir, shuffle=True)
bigram_val_ds = make_vectorized_loader(val_dir)
bigram_test_ds = make_vectorized_loader(test_dir)

In [ ]:
x, y = next(iter(bigram_train_ds))
x.shape

In [ ]:
text_vectorization.get_vocabulary()[100:108]

In [ ]:
model, optimizer, loss_fn = build_linear_classifier(max_tokens, "bigram_classifier")
fit(
    model, optimizer, loss_fn,
    bigram_train_ds, bigram_val_ds,
    epochs=10, patience=2,
)

In [ ]:
test_loss, test_acc = evaluate(model, loss_fn, bigram_test_ds)
test_acc

### Sequence models

In [ ]:
max_length = 600
max_tokens = 30_000
text_vectorization = TextVectorization(
    max_tokens=max_tokens,
    output_mode="int",
    output_sequence_length=max_length,
)
text_vectorization.adapt(train_texts)

sequence_train_ds = make_vectorized_loader(train_dir, shuffle=True)
sequence_val_ds = make_vectorized_loader(val_dir)
sequence_test_ds = make_vectorized_loader(test_dir)

In [ ]:
x, y = next(iter(sequence_test_ds))
x.shape

In [ ]:
x

#### Training a recurrent model

In [ ]:
# PyTorch: reimplement the OneHotEncoding layer as an nn.Module using
# torch.nn.functional.one_hot (float output so it can feed the LSTM).
import torch.nn.functional as F

class OneHotEncoding(nn.Module):
    def __init__(self, depth):
        super().__init__()
        self.depth = depth

    def forward(self, inputs):
        return F.one_hot(inputs.long(), num_classes=self.depth).float()

one_hot_encoding = OneHotEncoding(max_tokens)

In [ ]:
x, y = next(iter(sequence_train_ds))
one_hot_encoding(x).shape

In [ ]:
# PyTorch: build the functional Keras model as an nn.Module. Bidirectional LSTM ->
# nn.LSTM(bidirectional=True); we take the last-timestep hidden state from both
# directions (concatenated). Sigmoid is dropped in favor of BCEWithLogitsLoss.
hidden_dim = 64

class LSTMWithOneHot(nn.Module):
    def __init__(self, max_tokens, max_length, hidden_dim):
        super().__init__()
        self.one_hot = OneHotEncoding(max_tokens)
        self.lstm = nn.LSTM(max_tokens, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(2 * hidden_dim, 1)
        self.name = "lstm_with_one_hot"

    def forward(self, inputs):
        x = self.one_hot(inputs)
        _, (h, _) = self.lstm(x)
        x = torch.cat((h[-2], h[-1]), dim=1)  # forward + backward final states
        x = self.dropout(x)
        return self.fc(x)

model = LSTMWithOneHot(max_tokens, max_length, hidden_dim)
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
print(model)

In [ ]:
# PyTorch: the original TensorFlow-on-T4 bug note doesn't apply here.
# This model is large (one-hot inputs are 30k-wide), so training is slow.

In [ ]:
fit(
    model, optimizer, loss_fn,
    sequence_train_ds, sequence_val_ds,
    epochs=10, patience=2,
)

In [ ]:
test_loss, test_acc = evaluate(model, loss_fn, sequence_test_ds)
test_acc

#### Understanding word embeddings

#### Using a word embedding

In [ ]:
# PyTorch: layers.Embedding(mask_zero=True) -> nn.Embedding(padding_idx=0), which
# zeros the padding row and excludes it from gradient updates.
hidden_dim = 64

class LSTMWithEmbedding(nn.Module):
    def __init__(self, max_tokens, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(max_tokens, hidden_dim, padding_idx=0)
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(2 * hidden_dim, 1)
        self.name = "lstm_with_embedding"

    def forward(self, inputs):
        x = self.embedding(inputs)
        _, (h, _) = self.lstm(x)
        x = torch.cat((h[-2], h[-1]), dim=1)
        x = self.dropout(x)
        return self.fc(x)

model = LSTMWithEmbedding(max_tokens, hidden_dim)
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()

In [ ]:
print(model)

In [ ]:
fit(
    model, optimizer, loss_fn,
    sequence_train_ds, sequence_val_ds,
    epochs=10, patience=2,
)
test_loss, test_acc = evaluate(model, loss_fn, sequence_test_ds)
test_acc

#### Pretraining a word embedding

In [ ]:
# PyTorch: reuse our TextVectorization with the existing vocabulary but no fixed
# output length. We add a list-returning tokenizer for the CBOW windowing below.
imdb_vocabulary = text_vectorization.get_vocabulary()
tokenize_no_padding = TextVectorization(
    vocabulary=imdb_vocabulary,
    output_mode="int",
)

def tokenize_to_ids(text):
    return [tokenize_no_padding.index.get(tok, 1) for tok in tokenize_no_padding._tokens(text)]

In [ ]:
# PyTorch: rebuild the tf.data windowing pipeline as a torch Dataset. For each
# document we slide a window of `window_size` tokens; the center token is the label
# and the surrounding `2 * context_size` tokens are the bag (context).
context_size = 4
window_size = 9

class CBOWDataset(Dataset):
    def __init__(self, directory, tokenize_fn):
        self.examples = []  # (bag_tensor, label)
        directory = pathlib.Path(directory)
        for category in ("neg", "pos"):
            for path in sorted((directory / category).glob("*.txt")):
                ids = tokenize_fn(open(path, "r").read())
                num_windows = max(len(ids) - context_size * 2, 0)
                for start in range(num_windows):
                    window = ids[start:start + window_size]
                    bag = window[:context_size] + window[context_size + 1:]
                    label = window[context_size]
                    self.examples.append((
                        torch.tensor(bag, dtype=torch.long), label
                    ))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]

cbow_dataset = CBOWDataset(imdb_extract_dir / "train", tokenize_to_ids)

In [ ]:
# PyTorch: the CBOW model averages the context embeddings then predicts the center
# token. The original used a Dense(max_tokens, "sigmoid") + sparse categorical
# crossentropy; we drop the activation and use CrossEntropyLoss (softmax + sparse CE).
hidden_dim = 64

class CBOWModel(nn.Module):
    def __init__(self, max_tokens, hidden_dim):
        super().__init__()
        self.embedding = nn.Embedding(max_tokens, hidden_dim)
        self.fc = nn.Linear(hidden_dim, max_tokens)

    def forward(self, inputs):
        x = self.embedding(inputs)
        x = x.mean(dim=1)  # GlobalAveragePooling1D
        return self.fc(x)

cbow_model = CBOWModel(max_tokens, hidden_dim)
cbow_optimizer = torch.optim.Adam(cbow_model.parameters())
cbow_loss_fn = nn.CrossEntropyLoss()

In [ ]:
print(cbow_model)

In [ ]:
# PyTorch: batch with a DataLoader and run a plain training loop (no validation).
cbow_loader = DataLoader(cbow_dataset, batch_size=1024, shuffle=True)
for epoch in range(4):
    cbow_model.train()
    for inputs, targets in cbow_loader:
        logits = cbow_model(inputs)
        loss = cbow_loss_fn(logits, targets)
        cbow_optimizer.zero_grad()
        loss.backward()
        cbow_optimizer.step()
    print(f"Epoch {epoch}: loss {loss.item():.4f}")

#### Using the pretrained embedding for classification

In [ ]:
# PyTorch: same architecture as before; we'll copy in the pretrained CBOW embedding.
model = LSTMWithEmbedding(max_tokens, hidden_dim)
model.name = "lstm_with_cbow"

In [ ]:
# PyTorch: copy the pretrained CBOW embedding matrix into the classifier's embedding.
with torch.no_grad():
    model.embedding.weight.copy_(cbow_model.embedding.weight)

In [ ]:
optimizer = torch.optim.Adam(model.parameters())
loss_fn = nn.BCEWithLogitsLoss()
fit(
    model, optimizer, loss_fn,
    sequence_train_ds, sequence_val_ds,
    epochs=10, patience=2,
)

In [ ]:
test_loss, test_acc = evaluate(model, loss_fn, sequence_test_ds)
test_acc